<a href="https://colab.research.google.com/github/alreemoeed/DL-labs/blob/main/Lab7_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [10]:
!pip -q install transformers sentencepiece torch


In [11]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [12]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [14]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [15]:
# Task 1 — Choose the most suitable Transformer model family for each task.
model_families = {
    "Sentiment classification": "Encoder-only",
    "Machine translation": "Encoder-decoder",
    "Open-ended text generation": "Decoder-only",
    "Named Entity Recognition": "Encoder-only",
    "Text summarization": "Encoder-decoder",
}

for task, family in model_families.items():
    print(f"{task}: {family}")


Sentiment classification: Encoder-only
Machine translation: Encoder-decoder
Open-ended text generation: Decoder-only
Named Entity Recognition: Encoder-only
Text summarization: Encoder-decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [16]:
# Task 2 — Predict sentiment labels and confidence scores using DistilBERT.
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

predictions = classifier(sentences)
for sentence, prediction in zip(sentences, predictions):
    print(f"Sentence: {sentence}")
    print(f"Predicted label: {prediction['label']}")
    print(f"Score: {prediction['score']:.4f}\n")


Sentence: The movie was excellent.
Predicted label: POSITIVE
Score: 0.9999

Sentence: The lecture was difficult to understand.
Predicted label: NEGATIVE
Score: 0.9997

Sentence: I enjoyed learning about transformers.
Predicted label: POSITIVE
Score: 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [17]:
# Task 3 — Summarize a short paragraph using the FLAN-T5 encoder-decoder model.
paragraph = (
    "Large language models are trained on large collections of text to learn "
    "patterns in language. They can support tasks such as translation, "
    "summarization, and answering questions. Different Transformer "
    "architectures are useful for different tasks: encoder-only models "
    "understand and classify text, encoder-decoder models transform text, "
    "and decoder-only models generate text one token at a time."
)

summary_prompt = "Summarize the following paragraph: " + paragraph
inputs = tokenizer(summary_prompt, return_tensors="pt", truncation=True)
summary_ids = model.generate(**inputs, max_new_tokens=60, do_sample=False)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original paragraph:", paragraph)
print("\nSummary:", summary)


Original paragraph: Large language models are trained on large collections of text to learn patterns in language. They can support tasks such as translation, summarization, and answering questions. Different Transformer architectures are useful for different tasks: encoder-only models understand and classify text, encoder-decoder models transform text, and decoder-only models generate text one token at a time.

Summary: Learn the language. Learn the transformer architecture.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer:** A decoder-only model is suitable for chatbots because it generates text by predicting the next token based on the conversation so far. It can use the user's message and earlier conversation as context to produce a relevant response. Generating tokens one at a time allows it to create flexible, open-ended replies instead of selecting from fixed answers. This makes it useful for conversations, questions, and follow-up responses.


# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [18]:
# Optional Challenge — Sampling can produce different continuations for the same prompt.
challenge_prompt = "Artificial intelligence can help people"

for run in range(1, 4):
    generated = generator(
        challenge_prompt,
        max_new_tokens=25,
        do_sample=True,
        temperature=0.9,
        top_p=0.9,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    print(f"Run {run}: {generated[0]['generated_text']}\n")


[transformers] Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Artificial intelligence can help people understand the world from a more basic level. The world of robots could also allow us to create artificial intelligence and even better be



[transformers] Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: Artificial intelligence can help people improve their lives by improving their emotional and psychological abilities.















Run 3: Artificial intelligence can help people solve difficult problems. But it also helps to make us more confident.














**Answer:** When sampling is enabled (`do_sample=True`), the model randomly selects the next token from a probability distribution instead of always choosing the same highest-probability token. Different token selections can lead to different continuations, even with an identical prompt. Setting a fixed random seed can make sampling more reproducible, while removing the seed allows the results to vary across runs.
